# Limpieza de datos
**Contenido**
1. Carga de datos
2. Exploración inicial
3. Limpieza de nombres de columnas
4. Limpieza de variables financieras
5. Formateo de CIIU y año de corte
6. Análisis de duplicados
7. Exploración de variables categóricas
8. Limpieza de `ciudad_domicilio`
9. Limpieza de `departamento_domicilio`
    - 9.1 Detección de variantes
    - 9.2 Normalización y corrección
    - 9.3 Corrección de errores cruzados ciudad-departamento y homologación DIVIPOLA
    - 9.4 Verificación de cruces anómalos
    - 9.5 Homónimos legítimos
    - 9.6 Verificación final
10. Consolidación de razón social por NIT
11. Revisión de activos totales en cero


## 1. Carga de datos

In [8]:
# librerías
import pandas as pd
import numpy as np

In [9]:
ruta = '../data/raw/10.000_Empresas_mas_Grandes_del_País_20260909.csv'
df = pd.read_csv(ruta)

print(df.shape)
df.head()

(40000, 14)


,NIT,RAZÓN SOCIAL,SUPERVISOR,REGIÓN,DEPARTAMENTO DOMICILIO,CIUDAD DOMICILIO,CIIU,MACROSECTOR,INGRESOS OPERACIONALES,GANANCIA (PÉRDIDA),TOTAL ACTIVOS,TOTAL PASIVOS,TOTAL PATRIMONIO,Año de Corte
0,"899,999,068",ECOPETROL S.A.,SUPERFINANCIERA,BOGOTÁ - CUNDINAMARCA,BOGOTA D.C.,"BOGOTA, D.C.",610,MINERO,$144.82,$33.41,$216.85,$125.81,$91.03,"2,022"
1,"900,112,515",REFINERIA DE CARTAGENA S.A.S.,SUPERSOCIEDADES,COSTA ATLÁNTICA,BOLIVAR,CARTAGENA,"1,921",MANUFACTURA,$27.86,$2.19,$42.84,$16.48,$26.36,"2,022"
2,"830,095,213",ORGANIZACIÓN TERPEL S.A.,SUPERFINANCIERA,BOGOTÁ - CUNDINAMARCA,BOGOTA D.C.,"BOGOTA, D.C.","4,661",COMERCIO,$23.60,$0.33,$7.48,$4.47,$3.01,"2,022"
3,"860,069,804",CARBONES DEL CERREJON LIMITED,SUPERSOCIEDADES,BOGOTÁ - CUNDINAMARCA,BOGOTA D.C.,"BOGOTA, D.C.",510,MINERO,$16.39,$6.05,$10.45,$9.00,$1.45,"2,022"
4,"800,021,308",DRUMMOND LTD,SUPERSOCIEDADES,BOGOTÁ - CUNDINAMARCA,BOGOTA D.C.,"BOGOTA, D.C.",510,MINERO,$15.27,$2.16,$14.27,$6.34,$7.93,"2,022"


## 2. Exploración inicial
Vistazo a la estructura y los tipos de datos antes de limpiar.

In [10]:
# Información inicial de las columnas
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   NIT                     40000 non-null  str  
 1   RAZÓN SOCIAL            40000 non-null  str  
 2   SUPERVISOR              40000 non-null  str  
 3   REGIÓN                  40000 non-null  str  
 4   DEPARTAMENTO DOMICILIO  40000 non-null  str  
 5   CIUDAD DOMICILIO        40000 non-null  str  
 6   CIIU                    40000 non-null  str  
 7   MACROSECTOR             40000 non-null  str  
 8   INGRESOS OPERACIONALES  40000 non-null  str  
 9   GANANCIA (PÉRDIDA)      40000 non-null  str  
 10  TOTAL ACTIVOS           40000 non-null  str  
 11  TOTAL PASIVOS           40000 non-null  str  
 12  TOTAL PATRIMONIO        40000 non-null  str  
 13  Año de Corte            40000 non-null  str  
dtypes: str(14)
memory usage: 4.3 MB


In [11]:
df.isna().sum()

NIT                       0
RAZÓN SOCIAL              0
SUPERVISOR                0
REGIÓN                    0
DEPARTAMENTO DOMICILIO    0
CIUDAD DOMICILIO          0
CIIU                      0
MACROSECTOR               0
INGRESOS OPERACIONALES    0
GANANCIA (PÉRDIDA)        0
TOTAL ACTIVOS             0
TOTAL PASIVOS             0
TOTAL PATRIMONIO          0
Año de Corte              0
dtype: int64

## 3. Limpieza de nombres de columnas
Se estandarizan los encabezados a `snake_case`, sin tildes ni caracteres especiales.

In [12]:
# Normalizar nombres de columnas a snake_case
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.replace(r"\s+", "_", regex=True)
)

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   nit                     40000 non-null  str  
 1   razon_social            40000 non-null  str  
 2   supervisor              40000 non-null  str  
 3   region                  40000 non-null  str  
 4   departamento_domicilio  40000 non-null  str  
 5   ciudad_domicilio        40000 non-null  str  
 6   ciiu                    40000 non-null  str  
 7   macrosector             40000 non-null  str  
 8   ingresos_operacionales  40000 non-null  str  
 9   ganancia_perdida        40000 non-null  str  
 10  total_activos           40000 non-null  str  
 11  total_pasivos           40000 non-null  str  
 12  total_patrimonio        40000 non-null  str  
 13  ano_de_corte            40000 non-null  str  
dtypes: str(14)
memory usage: 4.3 MB


## 4. Limpieza de variables financieras
Las columnas monetarias llegan como texto (con `$` y comas); se convierten a numérico.

In [14]:
# Conversión de variables financieras a numéricas
cols_financieras = [
    "ingresos_operacionales",
    "ganancia_perdida",
    "total_activos",
    "total_pasivos",
    "total_patrimonio",
]

for col in cols_financieras:
    # Eliminamos el signo $, comas, y espacios, luego convertimos a numérico
    df[col] = df[col].astype(str).str.replace('$', '', regex=False)
    df[col] = df[col].str.replace(',', '', regex=False)
    df[col] = pd.to_numeric(df[col].str.strip(), errors='coerce')

display(df[cols_financieras].head())

,ingresos_operacionales,ganancia_perdida,total_activos,total_pasivos,total_patrimonio
0,144.82,33.41,216.85,125.81,91.03
1,27.86,2.19,42.84,16.48,26.36
2,23.60,0.33,7.48,4.47,3.01
3,16.39,6.05,10.45,9.00,1.45
4,15.27,2.16,14.27,6.34,7.93


## 5. Formateo de CIIU y año de corte

In [15]:
df['ciiu'] = (
    df['ciiu']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.split('.').str[0]
    .str.strip()
    .str.zfill(4)
)
## ano_corte a int
df['ano_de_corte'] = df['ano_de_corte'].astype(str).str.replace(',', '', regex=False)
df['ano_de_corte'] = df['ano_de_corte'].astype(int)

In [16]:
print(df['ciiu'].head().tolist())

['0610', '1921', '4661', '0510', '0510']


In [17]:
# Revisar qué columnas siguen siendo de tipo object (texto) tras las conversiones anteriores
np.column_stack(df.select_dtypes(include='object').columns)

/tmp/ipykernel_6454/291450324.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  np.column_stack(df.select_dtypes(include='object').columns)


array([['nit', 'razon_social', 'supervisor', 'region',
        'departamento_domicilio', 'ciudad_domicilio', 'ciiu',
        'macrosector']], dtype='<U22')

## 6. Análisis de duplicados

In [18]:
# Análisis de duplicados
print(f'Cantidad de filas duplicadas: {df.duplicated().sum()}')

Cantidad de filas duplicadas: 0


In [19]:
# Duplicados considerando que cada empresa debería tener un único registro por año
dup_nit_ano = df.duplicated(subset=["nit", "ano_de_corte"]).sum()
print(f'Filas con NIT repetido en el mismo año (posible duplicado real): {dup_nit_ano}')

Filas con NIT repetido en el mismo año (posible duplicado real): 0


## 7. Exploración de variables categóricas

In [20]:
# Quitar comas de nit
df['nit'] = df['nit'].astype(str).str.replace(',', '', regex=False)

In [21]:
# Análisis del comportamiento de las columnas categóricas
cols_cat = ['razon_social', 'supervisor', 'region', 'departamento_domicilio', 'ciudad_domicilio', 'ciiu', 'macrosector']

for col in cols_cat:
    print(f'\nColumna: {col}')
    print(df[col].value_counts())


Columna: razon_social
razon_social
CONVIAS S.A.S.                                          8
RADIOLOGOS ASOCIADOS S.A.S.                             7
SUMOTO S.A.                                             7
POLIMIX CONCRETO COLOMBIA S.A.S.                        5
YOYO S.A.S.                                             5
                                                       ..
GRANADOS; GOMEZ Y COMPAÑIA S.A. E.S.P.                  1
COOPERATIVA DE TRANSPORTADORES DEL SARAVITA LIMITADA    1
COMPAÑIA MIENRA EL MANZANILLO S.A.S.                    1
ANDINA DE PETROLEOS  S.A.S.                             1
C I UNIPACK COLOMBIA S.A.S.                             1
Name: count, Length: 16406, dtype: int64

Columna: supervisor
supervisor
SUPERSOCIEDADES    34834
SUPERSALUD          2885
SUPERTRANSPORTE      867
SUPERVIGILANCIA      773
SUPERSERVICIOS       420
SUPERFINANCIERA      221
Name: count, dtype: int64

Columna: region
region
BOGOTÁ - CUNDINAMARCA    18637
ANTIOQUIA             

In [22]:
# Estadísticas descriptivas de las variables numéricas
df.describe()

,ingresos_operacionales,ganancia_perdida,total_activos,total_pasivos,total_patrimonio,ano_de_corte
count,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000
mean,0.167670,0.013160,0.218061,0.110702,0.107080,2022.500000
std,1.392991,0.254042,2.369618,1.345162,1.117528,1.118048
min,0.010000,-3.210000,0.000000,0.000000,-3.690000,2021.000000
25%,0.030000,0.000000,0.020000,0.010000,0.010000,2021.750000
50%,0.040000,0.000000,0.030000,0.020000,0.010000,2022.500000
75%,0.100000,0.010000,0.090000,0.050000,0.040000,2023.250000
max,144.820000,33.410000,216.850000,130.540000,91.030000,2024.000000


## 8. Limpieza de `ciudad_domicilio`

### 8.1 Detección de variantes

In [23]:
# Ver posibles variantes de ciudad_domicilio agrupando por prefijo

ciudades_unicas = pd.Series(df["ciudad_domicilio"].dropna().unique())

variantes_por_prefijo = (
    ciudades_unicas.groupby(ciudades_unicas.str[:3])
    .agg(list)
    .loc[lambda s: s.str.len() > 1]
)

for prefijo, lista in variantes_por_prefijo.items():
    print(f"Prefijo '{prefijo}': {lista}")

Prefijo 'AGU': ['AGUACHICA', 'AGUAZUL']
Prefijo 'ALB': ['ALBAN', 'ALBANIA']
Prefijo 'ANS': ['ANSERMA', 'ANSERMANUEVO']
Prefijo 'ARA': ['ARAUCA', 'ARATOCA']
Prefijo 'ARM': ['ARMENIA', 'ARMERO']
Prefijo 'BAR': ['BARRANQUILLA', 'BARBOSA', 'BARANOA', 'BARRANCABERMEJA', 'BARRANCAS']
Prefijo 'BEL': ['BELLO', 'BELEN']
Prefijo 'BUC': ['BUCARAMANGA', 'BUCARASICA']
Prefijo 'BUG': ['BUGA', 'BUGALAGRANDE']
Prefijo 'CAL': ['CALI', 'CALOTO', 'CALDAS', 'CALARCA', 'CALIMA']
Prefijo 'CAM': ['CAMPOALEGRE', 'CAMPO DE LA CRUZ', 'CAMPO ALEGRE']
Prefijo 'CAR': ['CARTAGENA', 'CARTAGO', 'CAREPA', 'CARMEN DE VIBORAL']
Prefijo 'CHA': ['CHAPARRAL', 'CHACHAGÜÍ']
Prefijo 'CHI': ['CHIA', 'CHINCHINA', 'CHIQUINQUIRA', 'CHIGORODO', 'CHIRIGUANA']
Prefijo 'CIE': ['CIENAGA DE ORO', 'CIENAGA']
Prefijo 'COR': ['COROZAL', 'CORRALES', 'CORDOBA']
Prefijo 'CUC': ['CUCUTA', 'CUCUNUBA', 'CUCUTILLA']
Prefijo 'DON': ['DONMATIAS', 'DON MATIAS']
Prefijo 'DOS': ['DOSQUEBRADAS', 'DOS QUEBRADAS']
Prefijo 'EL ': ['EL ZULIA', 'EL CARMEN 

### 8.2 Normalización y corrección

In [24]:
# 1. Normalización general (quita tildes y diéresis de forma vectorizada)
df["ciudad_domicilio"] = (
    df["ciudad_domicilio"]
    .astype(str)
    .str.strip()
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
    .str.upper()
)

# 2. Diccionario de corrección para inconsistencias detectadas
mapeo_ciudades = {
    # Aglutinaciones y separación de palabras
    "DON MATIAS": "DONMATIAS",
    "DOS QUEBRADAS": "DOSQUEBRADAS",
    "CAMPO ALEGRE": "CAMPOALEGRE",
    # Sufijos departamentales y aclaraciones
    "MAICAO GUAJIRA": "MAICAO",
    "MITU VAUPES": "MITU",
    "PUERTO GAITAN (MANACACIAS)": "PUERTO GAITAN",
    "LA PAZ (ROBLES)": "LA PAZ",
    "SAN ANDRES Y PROVIDENCIA": "SAN ANDRES",
    # Nombres oficiales largos vs. comunes
    "GUADALAJARA DE BUGA": "BUGA",
    "VILLA DE SAN DIEGO DE UBATE": "UBATE",
    "VILLA DE SAN DIEGO DE VILLA DE SAN DIEGO DE UBATE": "UBATE",
    "VILLA ROSARIO": "VILLA DEL ROSARIO",
    # Textos corruptos o truncados
    "SAN SEBASTIAN DE BUENAVISMAGDALENA": "SAN SEBASTIAN DE BUENAVISTA",
}

df["ciudad_domicilio"] = df["ciudad_domicilio"].replace(mapeo_ciudades)



### 8.3 Caso especial: Ferretería Central (Boyacá)
Un registro puntual quedó con el nombre del departamento pegado a la ciudad y a la razón social; se corrige a mano.

In [25]:
# 1. Localizar por coincidencia de la razón social
filtro_ferreteria = df["razon_social"].str.contains(
    "FERRETERIA CENTRAL", na=False
) & (df["departamento_domicilio"] == "BOYACA")

# 2. Corregir ciudad y departamento
df.loc[filtro_ferreteria, "ciudad_domicilio"] = "PUERTO BOYACA"
df.loc[filtro_ferreteria, "departamento_domicilio"] = "BOYACA"

# 3. Limpiar la razón social si absorbió el texto del departamento
df.loc[filtro_ferreteria, "razon_social"] = df.loc[
    filtro_ferreteria, "razon_social"
].str.replace(r"BOYACA$", "", regex=True).str.strip()

## 9. Limpieza de `departamento_domicilio`

### 9.1 Detección de variantes

In [26]:
# Ver posibles variantes de departamento_domicilio agrupando por prefijo
departamentos_unicos = pd.Series(df["departamento_domicilio"].dropna().unique())

variantes_por_prefijo = (
    departamentos_unicos.groupby(departamentos_unicos.str[:3])
    .agg(list)
    .loc[lambda s: s.str.len() >= 1]
)

for prefijo, lista in variantes_por_prefijo.items():
    print(f"Prefijo '{prefijo}': {lista}")

Prefijo 'AMA': ['AMAZONAS']
Prefijo 'ANT': ['ANTIOQUIA']
Prefijo 'ARA': ['ARAUCA']
Prefijo 'ARM': ['ARMENIA']
Prefijo 'ATL': ['ATLANTICO']
Prefijo 'BOG': ['BOGOTA D.C.']
Prefijo 'BOL': ['BOLIVAR']
Prefijo 'BOY': ['BOYACA']
Prefijo 'CAL': ['CALDAS']
Prefijo 'CAQ': ['CAQUETA']
Prefijo 'CAS': ['CASANARE']
Prefijo 'CAU': ['CAUCA']
Prefijo 'CES': ['CESAR']
Prefijo 'CHO': ['CHOCO']
Prefijo 'COR': ['CORDOBA']
Prefijo 'CUN': ['CUNDINAMARCA']
Prefijo 'GUA': ['GUAVIARE', 'GUAINIA', 'GUAJIRA']
Prefijo 'HUI': ['HUILA']
Prefijo 'LA ': ['LA GUAJIRA']
Prefijo 'MAG': ['MAGDALENA']
Prefijo 'MET': ['META']
Prefijo 'MON': ['MONTERIA']
Prefijo 'NAR': ['NARIÑO', 'NARINO']
Prefijo 'NOR': ['NORTE DE SANTANDER']
Prefijo 'PUT': ['PUTUMAYO']
Prefijo 'QUI': ['QUINDIO']
Prefijo 'RIS': ['RISARALDA']
Prefijo 'SAN': ['SANTANDER', 'SAN ANDRES Y PROVIDENCIA']
Prefijo 'SUC': ['SUCRE']
Prefijo 'TOL': ['TOLIMA']
Prefijo 'VAL': ['VALLE', 'VALLE DEL CAUCA', 'VALLEDUPAR']
Prefijo 'VAU': ['VAUPES']
Prefijo 'VIC': ['VICHADA']

### 9.2 Normalización y corrección

In [27]:
# 1. Normalizar a mayúsculas y remover caracteres especiales/eñes
df["departamento_domicilio"] = (
    df["departamento_domicilio"]
    .astype(str)
    .str.strip()
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
    .str.upper()
)

# 2. Reubicar ciudades que quedaron registradas en la columna de departamento
# Si ciudad_domicilio quedó vacía o alterada, se asegura el valor correcto
mask_armenia = df["departamento_domicilio"] == "ARMENIA"
df.loc[mask_armenia, "ciudad_domicilio"] = "ARMENIA"
df.loc[mask_armenia, "departamento_domicilio"] = "QUINDIO"

mask_monteria = df["departamento_domicilio"] == "MONTERIA"
df.loc[mask_monteria, "ciudad_domicilio"] = "MONTERIA"
df.loc[mask_monteria, "departamento_domicilio"] = "CORDOBA"

mask_valledupar = df["departamento_domicilio"] == "VALLEDUPAR"
df.loc[mask_valledupar, "ciudad_domicilio"] = "VALLEDUPAR"
df.loc[mask_valledupar, "departamento_domicilio"] = "CESAR"

# 3. Estandarizar nombres de departamentos duplicados
mapeo_deptos = {
    "VALLE": "VALLE DEL CAUCA",
    "GUAJIRA": "LA GUAJIRA",
}

df["departamento_domicilio"] = df["departamento_domicilio"].replace(mapeo_deptos)

### 9.3 Corrección de errores cruzados ciudad-departamento y homologación DIVIPOLA
Se reubican ciudades que habían quedado asociadas a un departamento incorrecto y se homologan nombres de ciudad a su denominación oficial DIVIPOLA.

In [28]:
# FASE 1: Corregir departamentos y errores tipográficos puntuales

# Errores territoriales donde la ciudad es evidente
df.loc[df["ciudad_domicilio"] == "MEDELLIN", "departamento_domicilio"] = (
    "ANTIOQUIA"
)
df.loc[df["ciudad_domicilio"] == "BARRANQUILLA", "departamento_domicilio"] = (
    "ATLANTICO"
)
df.loc[df["ciudad_domicilio"] == "BUCARAMANGA", "departamento_domicilio"] = (
    "SANTANDER"
)
df.loc[df["ciudad_domicilio"] == "PALMIRA", "departamento_domicilio"] = (
    "VALLE DEL CAUCA"
)
df.loc[df["ciudad_domicilio"] == "RIOHACHA", "departamento_domicilio"] = (
    "LA GUAJIRA"
)
df.loc[
    df["ciudad_domicilio"].isin(["CUCUTA", "OCANA", "BUCARASICA"]),
    "departamento_domicilio",
] = "NORTE DE SANTANDER"
df.loc[
    df["ciudad_domicilio"].isin(["FUNZA", "COTA", "MOSQUERA"]),
    "departamento_domicilio",
] = "CUNDINAMARCA"
df.loc[df["ciudad_domicilio"] == "LA TEBAIDA", "departamento_domicilio"] = (
    "QUINDIO"
)

# Caso especial Bogotá (forzar siempre a Bogotá D.C. en ambas columnas)
mask_bogota = df["ciudad_domicilio"].isin(
    ["BOGOTA D.C.", "BOGOTA, D.C.", "BOGOTA"]
)
df.loc[mask_bogota, ["ciudad_domicilio", "departamento_domicilio"]] = (
    "BOGOTA D.C."
)

# Caso especial Anserma en Valle -> Ansermanuevo
df.loc[
    (df["ciudad_domicilio"] == "ANSERMA")
    & (df["departamento_domicilio"] == "VALLE DEL CAUCA"),
    "ciudad_domicilio",
] = "ANSERMANUEVO"

# Typo en Risaralda: Satuario -> Santuario
df.loc[
    (df["ciudad_domicilio"] == "SATUARIO")
    & (df["departamento_domicilio"] == "RISARALDA"),
    "ciudad_domicilio",
] = "SANTUARIO"


# FASE 2: Homologación a nombres oficiales DIVIPOLA

homologacion_nombres = {
    # Ciudades principales y municipios con prefijos/sufijos omitidos
    "CALI": "SANTIAGO DE CALI",
    "CARTAGENA": "CARTAGENA DE INDIAS",
    "CUCUTA": "SAN JOSE DE CUCUTA",
    "BUGA": "GUADALAJARA DE BUGA",
    "UBATE": "VILLA DE SAN DIEGO DE UBATE",
    "CARMEN DE VIBORAL": "EL CARMEN DE VIBORAL",
    "TUMACO": "SAN ANDRES DE TUMACO",
    "SANTAFE DE ANTIOQUIA": "SANTA FE DE ANTIOQUIA",
    "MARIQUITA": "SAN SEBASTIAN DE MARIQUITA",
    "PIENDAMO": "PIENDAMO - TUNIA",
    "SINCE": "SAN LUIS DE SINCE",
    "TOLU": "SANTIAGO DE TOLU",
    "INIRIDA GUAINIA": "INIRIDA",
    "ANTIOQUIA": "SANTA FE DE ANTIOQUIA",
}

# Aplicar reemplazos directos
df["ciudad_domicilio"] = df["ciudad_domicilio"].replace(homologacion_nombres)

# Casos con desambiguación contextual:
# San Pedro en Antioquia -> San Pedro de los Milagros
df.loc[
    (df["ciudad_domicilio"] == "SAN PEDRO")
    & (df["departamento_domicilio"] == "ANTIOQUIA"),
    "ciudad_domicilio",
] = "SAN PEDRO DE LOS MILAGROS"

# Bolívar en Antioquia -> Ciudad Bolívar
df.loc[
    (df["ciudad_domicilio"] == "BOLIVAR")
    & (df["departamento_domicilio"] == "ANTIOQUIA"),
    "ciudad_domicilio",
] = "CIUDAD BOLIVAR"

# Santuario en Antioquia -> El Santuario (en Risaralda el nombre oficial es "Santuario", sin "El";
# por eso esta homologación se condiciona por departamento en vez de aplicarse a toda la columna)
df.loc[
    (df["ciudad_domicilio"] == "SANTUARIO")
    & (df["departamento_domicilio"] == "ANTIOQUIA"),
    "ciudad_domicilio",
] = "EL SANTUARIO"

### 9.4 Verificación de cruces anómalos
Se confirma que ya no existan las combinaciones ciudad-departamento imposibles detectadas inicialmente.

In [29]:
# 1. Comprobar los cruces absurdos específicos citados en el texto
cruces_anomalos_esperados = {
    "BOGOTA D.C.": ["ANTIOQUIA", "CUNDINAMARCA"],
    "MEDELLIN": ["MAGDALENA"],
    "BARRANQUILLA": ["BOLIVAR", "SAN ANDRES Y PROVIDENCIA"],
    "CUCUTA": ["SANTANDER"],
}

condiciones = []
for ciudad, deptos in cruces_anomalos_esperados.items():
    cond = (df["ciudad_domicilio"] == ciudad) & (
        df["departamento_domicilio"].isin(deptos)
    )
    condiciones.append(cond)

filtro_anomalias = pd.concat(condiciones, axis=1).any(axis=1)
df_anomalias_detectadas = df.loc[
    filtro_anomalias,
    ["nit", "razon_social", "ciudad_domicilio", "departamento_domicilio", "ano_de_corte"],
]

print(f"Total de filas con cruces anómalos detectadas: {len(df_anomalias_detectadas)}")
print(df_anomalias_detectadas.to_string(index=False))

# 2. Contar cuántas ciudades únicas aparecen en más de un departamento
ciudades_multidepartamento = (
    df.groupby("ciudad_domicilio")["departamento_domicilio"]
    .nunique()
    .loc[lambda x: x > 1]
)

print(f"\nTotal de nombres de ciudad asociados a >1 departamento: {len(ciudades_multidepartamento)}")
print(ciudades_multidepartamento.head(15))

Total de filas con cruces anómalos detectadas: 0
Empty DataFrame
Columns: [nit, razon_social, ciudad_domicilio, departamento_domicilio, ano_de_corte]
Index: []

Total de nombres de ciudad asociados a >1 departamento: 4
ciudad_domicilio
GRANADA     2
GUAMAL      2
LA UNION    3
RESTREPO    2
Name: departamento_domicilio, dtype: int64


### 9.5 Homónimos legítimos
Se revisan los nombres de ciudad que siguen asociados a más de un departamento, para confirmar que corresponden a municipios distintos y no a errores.

In [30]:
homonimos = ["EL SANTUARIO", "GRANADA", "GUAMAL", "LA UNION", "RESTREPO"]

print(
    df[df["ciudad_domicilio"].isin(homonimos)][
        ["ciudad_domicilio", "departamento_domicilio"]
    ]
    .drop_duplicates()
    .sort_values(by=["ciudad_domicilio", "departamento_domicilio"])
    .to_string(index=False)
)

ciudad_domicilio departamento_domicilio
    EL SANTUARIO              ANTIOQUIA
         GRANADA           CUNDINAMARCA
         GRANADA                   META
          GUAMAL              MAGDALENA
          GUAMAL                   META
        LA UNION              ANTIOQUIA
        LA UNION                 NARINO
        LA UNION        VALLE DEL CAUCA
        RESTREPO                   META
        RESTREPO        VALLE DEL CAUCA


### 9.6 Verificación final
Listado final de departamentos/entidades territoriales tras todas las correcciones de la sección 9.

In [31]:
# Verificar la lista final de departamentos/entidades territoriales
deptos_unicos = sorted(df["departamento_domicilio"].dropna().unique())
print(f"Total entidades territoriales encontradas: {len(deptos_unicos)}")
print(deptos_unicos)

Total entidades territoriales encontradas: 33
['AMAZONAS', 'ANTIOQUIA', 'ARAUCA', 'ATLANTICO', 'BOGOTA D.C.', 'BOLIVAR', 'BOYACA', 'CALDAS', 'CAQUETA', 'CASANARE', 'CAUCA', 'CESAR', 'CHOCO', 'CORDOBA', 'CUNDINAMARCA', 'GUAINIA', 'GUAVIARE', 'HUILA', 'LA GUAJIRA', 'MAGDALENA', 'META', 'NARINO', 'NORTE DE SANTANDER', 'PUTUMAYO', 'QUINDIO', 'RISARALDA', 'SAN ANDRES Y PROVIDENCIA', 'SANTANDER', 'SUCRE', 'TOLIMA', 'VALLE DEL CAUCA', 'VAUPES', 'VICHADA']


## 10. Consolidación de razón social por NIT
Un mismo NIT puede tener distintos nombres registrados a lo largo de los años; se conserva el más reciente como definitivo.

### 10.1 Detección de inconsistencias

In [32]:
# 1. Agrupar por NIT y contar cuántas razones sociales distintas tiene asociadas
nits_con_varias_razones = (
    df.groupby("nit")["razon_social"]
    .nunique()
    .loc[lambda x: x > 1]
)

print(f"Total de NITs con más de una razón social: {len(nits_con_varias_razones)}")

# 2. Filtrar el DataFrame original para inspeccionar los casos conflictivos
casos_inconsistentes = (
    df[df["nit"].isin(nits_con_varias_razones.index)][
        ["nit", "razon_social", "ano_de_corte"]
    ]
    .drop_duplicates()
    .sort_values(by=["nit", "ano_de_corte"])
)

# Mostrar los primeros casos
print("\nEjemplo de empresas con nombres cambiantes:")
print(casos_inconsistentes.head(20).to_string(index=False))

Total de NITs con más de una razón social: 2230

Ejemplo de empresas con nombres cambiantes:
      nit                                                                                 razon_social  ano_de_corte
800000118 EMPRESA SOCIAL DEL ESTADO HOSPITAL  DEPARTAMENTAL UNIVERSITARIO DEL QUINDIO SAN JUAN DE DIOS          2021
800000118 EMPRESA SOCIAL DEL ESTADO HOSPITAL  DEPARTAMENTAL UNIVERSITARIO DEL QUINDIO SAN JUAN DE DIOS          2022
800000118  EMPRESA SOCIAL DEL ESTADO HOSPITAL DEPARTAMENTAL UNIVERSITARIO DEL QUINDIO SAN JUAN DE DIOS          2023
800002942                                                                                ITELCA S.A.S.          2021
800002942                                                                                ITELCA S.A.S.          2022
800002942                                                                               ITELCA  S.A.S.          2023
800002942                                                                               

### 10.2 Corrección: se conserva el nombre más reciente

In [33]:
# 1. Normalizar espacios en blanco internos y bordes
df["razon_social"] = (
    df["razon_social"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# 2. Obtener el nombre del año más reciente por NIT
nombre_mas_reciente = (
    df.sort_values(by=["nit", "ano_de_corte"], ascending=[True, True])
    .groupby("nit")["razon_social"]
    .last()
)

# 3. Sobreescribir
df["razon_social"] = df["nit"].map(nombre_mas_reciente)

### 10.3 Verificación

In [34]:
# Verificar que ya no existan NIT con más de un nombre asociado
print("NITs con más de un nombre:", (df.groupby("nit")["razon_social"].nunique() > 1).sum())

NITs con más de un nombre: 0


## 11. Revisión de activos totales en cero

In [35]:
# Contar cuántas empresas quedaron con total_activos en 0.0
activos_cero = (df["total_activos"] == 0).sum()
print(f"Empresas con activos reportados en 0: {activos_cero}")

# Revisar si se trata de empresas en proceso de liquidación o balance vacío
if activos_cero > 0:
    print(
        df[df["total_activos"] == 0][
            ["nit", "razon_social", "ano_de_corte", "ingresos_operacionales"]
        ].head(10)
    )

Empresas con activos reportados en 0: 1137
            nit                                       razon_social  \
1688  900838709                      HUNZAGRO FERTILIZANTES S.A.S.   
1951  900334644                                 NET COMPANY S.A.S.   
2390  901253588                                      INCOMM S.A.S.   
2562  890928110                              SURTIALIMENTOS S.A.S.   
2979  900627999           CAMERON COLOMBIA LLC SUCURSAL COLOMBIANA   
3144  830056174                         SERVIMAX Y COMPAÑIA S.A.S.   
3310  901435143  COMERCIALIZADORA INTERNACIONAL PACIFIC PREMIUM...   
3394  800215548                        DISTRIBUCIONES DISAY S.A.S.   
3481  800022862          FRUTAS DE EXPORTACIO S.A.S. FRUTEX S.A.S.   
3589  900618704                                      ICONYK S.A.S.   

      ano_de_corte  ingresos_operacionales  
1688          2022                    0.15  
1951          2022                    0.12  
2390          2022                    0.10  
2562  

In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   nit                     40000 non-null  str    
 1   razon_social            40000 non-null  str    
 2   supervisor              40000 non-null  str    
 3   region                  40000 non-null  str    
 4   departamento_domicilio  40000 non-null  str    
 5   ciudad_domicilio        40000 non-null  str    
 6   ciiu                    40000 non-null  object 
 7   macrosector             40000 non-null  str    
 8   ingresos_operacionales  40000 non-null  float64
 9   ganancia_perdida        40000 non-null  float64
 10  total_activos           40000 non-null  float64
 11  total_pasivos           40000 non-null  float64
 12  total_patrimonio        40000 non-null  float64
 13  ano_de_corte            40000 non-null  int64  
dtypes: float64(5), int64(1), object(1), str(7)
memory

In [ ]:
# Exportar resultados
df_limpio = df.copy()
df_limpio.to_csv('../data/proccesed/10.000_Empresas_mas_Grandes_del_País_proccesed.csv', index=False)